# S14 EEG Analysis from MATLAB `.mat` File

This notebook loads an EEG recording from `data/S14_EEG.mat` and performs:

- MATLAB variable inspection
- EEG matrix detection and orientation correction
- Descriptive statistics
- Raw EEG visualization
- Welch power spectral density analysis
- Delta, theta, alpha, beta, and gamma band-power estimation
- Saving CSV, JSON, and PNG outputs in the `results/` folder

**Important:** The provenance and authenticity of the recording must be independently verified. No claim that the data were collected from human participants should be made without appropriate documentation.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import loadmat

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

In [ ]:
# The notebook is inside notebooks/, so .. refers to the project root.
PROJECT_ROOT = Path('..')
MAT_PATH = PROJECT_ROOT / 'data' / 'S14_EEG.mat'
RESULTS_DIR = PROJECT_ROOT / 'results'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not MAT_PATH.exists():
    raise FileNotFoundError(
        f'File not found: {MAT_PATH}. '
        'Place S14_EEG.mat inside the data/ folder.'
    )

print(f'Input file: {MAT_PATH}')
print(f'Results directory: {RESULTS_DIR}')

## Load the MATLAB file

The code first lists all variables stored in the MAT file. Common EEG variable names include `data`, `eeg_data`, `EEG`, `signal`, and `X`.

In [ ]:
mat_contents = loadmat(MAT_PATH, squeeze_me=True, struct_as_record=False)

public_variables = {
    key: value
    for key, value in mat_contents.items()
    if not key.startswith('__')
}

print('Variables found in the MAT file:')
for key, value in public_variables.items():
    try:
        shape = np.asarray(value).shape
    except Exception:
        shape = 'unknown'
    print(f'  {key}: shape={shape}, type={type(value).__name__}')

In [ ]:
def find_variable(contents, candidate_names):
    for name in candidate_names:
        if name in contents:
            return contents[name], name

    lower_map = {key.lower(): key for key in contents.keys()}
    for name in candidate_names:
        if name.lower() in lower_map:
            original_key = lower_map[name.lower()]
            return contents[original_key], original_key

    return None, None


data_raw, data_variable_name = find_variable(
    mat_contents,
    ['data', 'eeg_data', 'EEG', 'signal', 'signals', 'X']
)

fs_raw, fs_variable_name = find_variable(
    mat_contents,
    ['fs', 'srate', 'sampling_rate', 'sample_rate', 'sampling_frequency']
)

labels_raw, labels_variable_name = find_variable(
    mat_contents,
    ['labels', 'channel_labels', 'chanlabels', 'channels', 'ch_names']
)

if data_raw is None:
    raise KeyError(
        'No EEG data variable was detected. Expected one of: '
        'data, eeg_data, EEG, signal, signals, X.'
    )

if fs_raw is None:
    raise KeyError(
        'No sampling-rate variable was detected. Expected one of: '
        'fs, srate, sampling_rate, sample_rate, sampling_frequency.'
    )

print(f'Data variable: {data_variable_name}')
print(f'Sampling-rate variable: {fs_variable_name}')
print(f'Labels variable: {labels_variable_name}')

In [ ]:
data_uv = np.asarray(data_raw, dtype=float)
data_uv = np.squeeze(data_uv)

if data_uv.ndim != 2:
    raise ValueError(
        f'EEG data must be a 2D matrix, but received shape {data_uv.shape}.'
    )

fs = float(np.asarray(fs_raw).squeeze())

if fs <= 0:
    raise ValueError(f'Sampling rate must be positive, received {fs}.')

def clean_label(item):
    value = item
    if isinstance(value, np.ndarray):
        value = value.squeeze()
        if value.size == 1:
            value = value.item()
    return str(value)


if labels_raw is not None:
    labels_array = np.asarray(labels_raw, dtype=object).squeeze()
    labels = [clean_label(item) for item in np.atleast_1d(labels_array)]
else:
    labels = []

# Convert the matrix to channels x samples.
if labels and len(labels) == data_uv.shape[1] and len(labels) != data_uv.shape[0]:
    data_uv = data_uv.T
elif not labels and data_uv.shape[0] > data_uv.shape[1]:
    data_uv = data_uv.T

n_channels, n_samples = data_uv.shape

if len(labels) != n_channels:
    labels = [f'Channel_{i + 1}' for i in range(n_channels)]

time = np.arange(n_samples) / fs
duration_seconds = n_samples / fs

print('Final data shape: channels x samples =', data_uv.shape)
print('Sampling rate:', fs, 'Hz')
print('Duration:', duration_seconds, 'seconds')
print('Channel labels:', labels)

## Descriptive statistics

In [ ]:
summary = pd.DataFrame({
    'channel': labels,
    'sampling_rate_hz': fs,
    'n_samples': n_samples,
    'duration_seconds': duration_seconds,
    'mean_uv': np.mean(data_uv, axis=1),
    'std_uv': np.std(data_uv, axis=1, ddof=1),
    'minimum_uv': np.min(data_uv, axis=1),
    'maximum_uv': np.max(data_uv, axis=1),
    'rms_uv': np.sqrt(np.mean(data_uv ** 2, axis=1)),
})

summary

## Plot raw EEG signals

The code assumes that the signal values are already expressed in microvolts. If the MAT file uses another unit, update the conversion before interpreting the results.

In [ ]:
fig, axes = plt.subplots(
    n_channels,
    1,
    figsize=(14, max(4, 2.5 * n_channels)),
    sharex=True
)

axes = np.atleast_1d(axes)

for i, ax in enumerate(axes):
    ax.plot(time, data_uv[i], linewidth=0.6)
    ax.set_ylabel('µV')
    ax.set_title(labels[i])
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Time (seconds)')
fig.suptitle('S14 EEG Raw Signals', fontsize=16)
plt.tight_layout()

raw_figure_path = RESULTS_DIR / 'S14_raw_eeg.png'
plt.savefig(raw_figure_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved: {raw_figure_path}')

## Welch power spectral density

In [ ]:
psd_results = {}

fig, axes = plt.subplots(
    n_channels,
    1,
    figsize=(12, max(4, 2.5 * n_channels)),
    sharex=True
)

axes = np.atleast_1d(axes)

for i, ax in enumerate(axes):
    frequencies, psd = signal.welch(
        data_uv[i],
        fs=fs,
        nperseg=min(n_samples, max(8, int(fs * 4)))
    )

    psd_results[labels[i]] = {
        'frequencies': frequencies,
        'psd': psd
    }

    mask = frequencies <= 50
    ax.semilogy(frequencies[mask], psd[mask], linewidth=1)
    ax.set_ylabel('PSD')
    ax.set_title(labels[i])
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Frequency (Hz)')
fig.suptitle('S14 EEG Power Spectral Density', fontsize=16)
plt.tight_layout()

psd_figure_path = RESULTS_DIR / 'S14_psd.png'
plt.savefig(psd_figure_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved: {psd_figure_path}')

In [ ]:
dominant_frequency_rows = []

for label, result in psd_results.items():
    frequencies = result['frequencies']
    psd = result['psd']
    valid = (frequencies >= 1) & (frequencies <= 45)

    if np.any(valid):
        peak_index = np.argmax(psd[valid])
        dominant_frequency = frequencies[valid][peak_index]
    else:
        dominant_frequency = np.nan

    dominant_frequency_rows.append({
        'channel': label,
        'dominant_frequency_hz': dominant_frequency
    })

dominant_frequency_table = pd.DataFrame(dominant_frequency_rows)
dominant_frequency_table

## EEG frequency-band power

Bands used here are conventional exploratory ranges:

- Delta: 1–4 Hz
- Theta: 4–8 Hz
- Alpha: 8–13 Hz
- Beta: 13–30 Hz
- Gamma: 30–45 Hz

In [ ]:
def bandpower(x, fs, low, high):
    frequencies, psd = signal.welch(
        x,
        fs=fs,
        nperseg=min(len(x), max(8, int(fs * 4)))
    )

    mask = (frequencies >= low) & (frequencies <= high)

    if np.sum(mask) < 2:
        return np.nan

    return np.trapezoid(psd[mask], frequencies[mask])


bands = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 45),
}

band_rows = []

for i, label in enumerate(labels):
    row = {'channel': label}
    absolute_powers = {}

    for band_name, (low, high) in bands.items():
        power = bandpower(data_uv[i], fs, low, high)
        absolute_powers[band_name] = power
        row[f'{band_name}_power_uv_squared'] = power

    valid_powers = [value for value in absolute_powers.values() if np.isfinite(value)]
    total_power = sum(valid_powers)

    for band_name, power in absolute_powers.items():
        row[f'{band_name}_relative_power'] = (
            power / total_power
            if total_power > 0 and np.isfinite(power)
            else np.nan
        )

    band_rows.append(row)

bandpower_table = pd.DataFrame(band_rows)
bandpower_table

In [ ]:
final_summary = (
    summary
    .merge(dominant_frequency_table, on='channel', how='left')
    .merge(bandpower_table, on='channel', how='left')
)

summary_path = RESULTS_DIR / 'S14_channel_summary.csv'
final_summary.to_csv(summary_path, index=False)

metadata_for_json = {
    'input_file': str(MAT_PATH),
    'data_variable': data_variable_name,
    'sampling_rate_variable': fs_variable_name,
    'labels_variable': labels_variable_name,
    'sampling_rate_hz': fs,
    'n_channels': n_channels,
    'n_samples': n_samples,
    'duration_seconds': duration_seconds,
    'channels': labels,
    'unit_assumption': 'microvolts; verify from the MAT-file documentation',
    'provenance_warning': 'Verify provenance and authenticity independently; do not claim human data without documentation.'
}

metadata_path = RESULTS_DIR / 'S14_mat_metadata.json'
with metadata_path.open('w', encoding='utf-8') as f:
    json.dump(metadata_for_json, f, ensure_ascii=False, indent=2)

print(f'Saved: {summary_path}')
print(f'Saved: {metadata_path}')
print(f'Saved: {raw_figure_path}')
print(f'Saved: {psd_figure_path}')

## Interpretation and limitations

This notebook provides a reproducible signal-processing workflow. The output is not, by itself, evidence of a clinical, cognitive, or neurocognitive effect.

ERP analyses such as N400 or P600 require event markers, repeated trials, preprocessing, baseline correction, condition labels, and appropriate statistical analysis. The provenance and authenticity of the MAT file must also be documented before making claims about human participants.